In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,roc_auc_score
from sklearn.svm import SVC
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')

In [2]:
# load cleaned dataset
data = pd.read_csv('/content/drive/MyDrive/DS & ML/diabetic_data_clean.csv')
data.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,total_visits,total_med_types
0,Caucasian,Female,85.0,Urgent,Discharged to home,Transfer-hospital,13,MC,InternalMedicine,68,...,0,0,0,0,0,Ch,Yes,0,0,2
1,Caucasian,Female,95.0,Elective,Transferred-SNF,Transfer-hospital,12,MC,InternalMedicine,33,...,0,0,0,0,0,Ch,Yes,0,0,2
2,Caucasian,Male,45.0,Emergency,Discharged to home,Emergency Room,1,MC,InternalMedicine,51,...,0,0,0,0,0,Ch,Yes,0,0,2
3,AfricanAmerican,Female,45.0,Emergency,Discharged to home,Emergency Room,9,MC,InternalMedicine,47,...,0,0,0,0,0,No,Yes,0,0,1
4,Caucasian,Male,55.0,Urgent,Discharged to home,Clinic Referral,3,MC,InternalMedicine,31,...,0,0,0,0,0,No,Yes,0,0,1


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69980 entries, 0 to 69979
Data columns (total 44 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   race                      69980 non-null  object 
 1   gender                    69980 non-null  object 
 2   age                       69980 non-null  float64
 3   admission_type_id         69980 non-null  object 
 4   discharge_disposition_id  69980 non-null  object 
 5   admission_source_id       69980 non-null  object 
 6   time_in_hospital          69980 non-null  int64  
 7   payer_code                69980 non-null  object 
 8   medical_specialty         69980 non-null  object 
 9   num_lab_procedures        69980 non-null  int64  
 10  num_procedures            69980 non-null  int64  
 11  num_medications           69980 non-null  int64  
 12  diag_1                    69980 non-null  object 
 13  diag_2                    69980 non-null  object 
 14  diag_3

In [4]:
data.isnull().sum()

,0
race,0
gender,0
age,0
admission_type_id,0
discharge_disposition_id,0
admission_source_id,0
time_in_hospital,0
payer_code,0
medical_specialty,0
num_lab_procedures,0


In [5]:
# Preprocess data
selected_features = [
    'race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures' , 'num_procedures',
    'num_medications', 'number_diagnoses', 'admission_type_id', 'discharge_disposition_id',
    'admission_source_id','medical_specialty', 'payer_code', 'change', 'diabetesMed',
    'total_visits', 'total_med_types'
]
X = data[selected_features]
y = data[['readmitted']]

### ENCODING

In [6]:
# Label encoding
le = LabelEncoder()
X['diabetesMed'] = le.fit_transform(X['diabetesMed'])
X['change'] = le.fit_transform(X['change'])
X['gender'] = le.fit_transform(X['gender'])
X['medical_specialty'] = le.fit_transform(X['medical_specialty'])
X['payer_code'] = le.fit_transform(X['payer_code'])


In [7]:
# One-hot encoding on columns with small number of unique values
low_card_cols = [
    'race', 'admission_type_id', 'discharge_disposition_id',
    'admission_source_id'
]
X = pd.get_dummies(X, columns=low_card_cols, drop_first=True)

In [14]:
# Scaling numeric columns
numeric_cols =['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications','number_diagnoses']
scaler = MinMaxScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])


In [15]:
# Split data
train_X, val_X, train_y, val_y = train_test_split(
    X, y, random_state=0, train_size=0.8
)



## **KNN**

In [16]:
# Train model
kneighbor = KNeighborsClassifier(n_neighbors=2, metric='euclidean', weights='uniform', algorithm='auto', leaf_size=50, p=2)
kneighbor.fit(train_X, train_y)


KNeighborsClassifier(leaf_size=50, metric='euclidean', n_neighbors=2)

In [17]:
# Evaluate model
val_prediction = kneighbor.predict(val_X)
y_pred_proba = kneighbor.predict_proba(val_X)[:,1]
accuracy = accuracy_score(val_y, val_prediction)
print(f'Model accuracy: {accuracy}')


Model accuracy: 0.9042583595312946


In [18]:
print(confusion_matrix(val_y, val_prediction))
print(classification_report(val_y, val_prediction))

[[12633   102]
 [ 1238    23]]
              precision    recall  f1-score   support

           0       0.91      0.99      0.95     12735
           1       0.18      0.02      0.03      1261

    accuracy                           0.90     13996
   macro avg       0.55      0.51      0.49     13996
weighted avg       0.85      0.90      0.87     13996



In [19]:
auc = roc_auc_score(val_y, y_pred_proba)
print(auc)

0.531506613026412


In [39]:
# Save model
joblib.dump(kneighbor, 'diabetes_readmission_model_knn.pkl')

['diabetes_readmission_model_knn.pkl']

In [31]:
# OOP APPROACH
class ReadmissionPrediction:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None
        self.X = None
        self.y = None
        self.train_X = None
        self.val_X = None
        self.train_y = None
        self.val_y = None
        self.model = None

    def load_data(self):
        self.data = pd.read_csv('/content/drive/MyDrive/DS & ML/diabetic_data_clean.csv')

    def preprocess_data(self):
        selected_features = [
       'race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures' , 'num_procedures',
       'num_medications', 'number_diagnoses', 'admission_type_id', 'discharge_disposition_id',
       'admission_source_id','medical_specialty', 'payer_code', 'change', 'diabetesMed',
       'total_visits', 'total_med_types'
        ]
        self.X = self.data[selected_features]
        self.y = self.data[['readmitted']] # Corrected column name here

        # Encoding categorical variables
        le = LabelEncoder()
        self.X['diabetesMed'] = le.fit_transform(self.X['diabetesMed'])
        self.X['change'] = le.fit_transform(self.X['change'])
        self.X['gender'] = le.fit_transform(self.X['gender'])
        self.X['medical_specialty'] = le.fit_transform(self.X['medical_specialty'])
        self.X['payer_code'] = le.fit_transform(self.X['payer_code'])

        # One-hot encoding on columns with small number of unique values
        low_card_cols = [
        'race', 'admission_type_id', 'discharge_disposition_id',
       'admission_source_id'
        ]
        self.X = pd.get_dummies(self.X, columns=low_card_cols, drop_first=True) # Assign back to self.X

        # Scaling numerical variables
        numeric_cols =['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
        'number_diagnoses']
        scaler = MinMaxScaler()
        self.X[numeric_cols] = scaler.fit_transform(self.X[numeric_cols]) # Use scaler object and self.X on both sides

    def split_data(self):
        self.train_X, self.val_X, self.train_y, self.val_y = train_test_split(
            self.X, self.y, random_state=0, train_size=0.8
        )

    def train_model(self):
        self.model = KNeighborsClassifier(n_neighbors=10)
        self.model.fit(self.train_X, self.train_y)

    def evaluate_model(self):
        val_prediction = self.model.predict(self.val_X)
        accuracy = accuracy_score(self.val_y, val_prediction)
        print(f'Model accuracy: {accuracy}')
        y_pred_proba = self.model.predict_proba(self.val_X)[:,1]
        auc = roc_auc_score(self.val_y, y_pred_proba)
        print(f'Model auc score: {auc}')
        return accuracy, auc

    def save_model(self, model_path):
        joblib.dump(self.model, model_path)

    def load_model(self, model_path):
        self.model = joblib.load(model_path)


In [38]:
# OOP APPROACH Usage
readmission = ReadmissionPrediction('/content/drive/MyDrive/DS & ML/diabetic_data_clean.csv')
readmission.load_data()
readmission.preprocess_data()
readmission.split_data()
readmission.train_model()
accuracy = readmission.evaluate_model()

# Save the model
readmission.save_model('diabetes_readmission_model_knn_oop.pkl')

Model accuracy: 0.9100457273506716
Model auc score: 0.5598355048794013


In [35]:
# PROCEDURAL APPROACH
def load_data(file_path):
    data = pd.read_csv(file_path)
    return data

def preprocess_data(data):
    selected_features = [
        'race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures' , 'num_procedures',
        'num_medications', 'number_diagnoses', 'admission_type_id', 'discharge_disposition_id',
        'admission_source_id','medical_specialty', 'payer_code', 'change', 'diabetesMed',
        'total_visits', 'total_med_types'
    ]
    X = data[selected_features]
    y = data[['readmitted']]

    # Label encoding
    le = LabelEncoder()
    X['diabetesMed'] = le.fit_transform(X['diabetesMed'])
    X['change'] = le.fit_transform(X['change'])
    X['gender'] = le.fit_transform(X['gender'])
    X['medical_specialty'] = le.fit_transform(X['medical_specialty'])
    X['payer_code'] = le.fit_transform(X['payer_code'])

    # One-hot encoding on columns with small number of unique values
    low_card_cols = [
        'race', 'admission_type_id', 'discharge_disposition_id',
        'admission_source_id'
    ]
    X = pd.get_dummies(X, columns=low_card_cols, drop_first=True)

    # Scaling numerical variables
    numeric_cols =['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
    'number_diagnoses']
    scaler = MinMaxScaler()
    X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

    return X, y

def split_data(X, y):
    train_X, val_X, train_y, val_y = train_test_split(
        X, y, random_state=0, train_size=0.8
    )
    return train_X, val_X, train_y, val_y

def train_model(train_X, train_y):
    model = KNeighborsClassifier(n_neighbors=2)
    model.fit(train_X, train_y)
    return model

def evaluate_model(model, val_X, val_y):
    val_prediction = model.predict(val_X)
    accuracy = accuracy_score(val_y, val_prediction)
    print(f'Model accuracy: {accuracy}')

    auc = roc_auc_score(val_y, val_prediction)
    print(f'Model auc score: {auc}')
    return accuracy, auc

def save_model(model, model_path):
    joblib.dump(model, model_path)

def load_model(model_path):
    model = joblib.load(model_path)
    return model

In [37]:
# PROCEDURAL Usage
file_path = '/content/drive/MyDrive/DS & ML/diabetic_data_clean.csv'
data = load_data(file_path)
X, y = preprocess_data(data)
train_X, val_X, train_y, val_y = split_data(X, y)
model = train_model(train_X, train_y)
accuracy, auc = evaluate_model(model, val_X, val_y)
save_model(model=model, model_path='diabetes_readmission_model_knn_procedural.pkl')

Model accuracy: 0.9042583595312946
Model auc score: 0.5051150348079422


## **DECISION TREE**

In [42]:
# OOP APPROACH
class DiabetesReadmission:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None
        self.X = None
        self.y = None
        self.train_X = None
        self.val_X = None
        self.train_y = None
        self.val_y = None
        self.model = None

    def load_data(self):
        self.data = pd.read_csv(self.file_path)

    def preprocess_data(self):
        selected_features = [
            'race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures' , 'num_procedures',
            'num_medications', 'number_diagnoses', 'admission_type_id', 'discharge_disposition_id',
            'admission_source_id','medical_specialty', 'payer_code', 'change', 'diabetesMed',
            'total_visits', 'total_med_types'
        ]
        self.X = self.data[selected_features]
        self.y = self.data[['readmitted']]

        # Label encoding for categorical columns
        le = LabelEncoder()
        self.X['diabetesMed'] = le.fit_transform(self.X['diabetesMed'])
        self.X['change'] = le.fit_transform(self.X['change'])
        self.X['gender'] = le.fit_transform(self.X['gender'])
        self.X['medical_specialty'] = le.fit_transform(self.X['medical_specialty'])
        self.X['payer_code'] = le.fit_transform(self.X['payer_code'])

        # One-hot encoding for low cardinality columns
        low_card_cols = [
            'race', 'admission_type_id', 'discharge_disposition_id',
            'admission_source_id'
        ]
        self.X = pd.get_dummies(self.X, columns=low_card_cols, drop_first=True)

        # Scaling numerical variables
        numeric_cols =['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
        'number_diagnoses']
        scaler = MinMaxScaler()
        self.X[numeric_cols] = scaler.fit_transform(self.X[numeric_cols])


    def split_data(self):
        self.train_X, self.val_X, self.train_y, self.val_y = train_test_split(
            self.X, self.y, random_state=0, train_size=0.8
        )

    def train_model(self):
        self.model = DecisionTreeClassifier(random_state=42)
        self.model.fit(self.train_X, self.train_y)

    def evaluate_model(self):
        val_prediction = self.model.predict(self.val_X)
        accuracy = accuracy_score(self.val_y, val_prediction)
        print(f'Model accuracy: {accuracy}')
        y_pred_proba = self.model.predict_proba(self.val_X)[:,1]
        auc = roc_auc_score(self.val_y, y_pred_proba)
        print(f'Model auc score: {auc}')
        return accuracy, auc

    def save_model(self, model_path):
        joblib.dump(self.model, model_path)

    def load_model(self, model_path):
        self.model = joblib.load(model_path)


In [43]:
# OOP APPROACH Usage
readmission = DiabetesReadmission('/content/drive/MyDrive/DS & ML/diabetic_data_clean.csv')
readmission.load_data()
readmission.preprocess_data()
readmission.split_data()
readmission.train_model()
accuracy, auc = readmission.evaluate_model()

# Save the model
readmission.save_model('diabetes_readmission_model_dt_oop.pkl')

Model accuracy: 0.8222349242640754
Model auc score: 0.5193457059618584


In [ ]:
# PROCEDURAL APPROACH
def load_data(file_path):
    return pd.read_csv(file_path)

def preprocess_data(data):
    selected_features = [
        'race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures' , 'num_procedures',
        'num_medications', 'number_diagnoses', 'admission_type_id', 'discharge_disposition_id',
        'admission_source_id','medical_specialty', 'payer_code', 'change', 'diabetesMed',
        'total_visits', 'total_med_types'
    ]
    X = data[selected_features]
    y = data[['readmitted']]

    # Label encoding
    le = LabelEncoder()
    X['diabetesMed'] = le.fit_transform(X['diabetesMed'])
    X['change'] = le.fit_transform(X['change'])
    X['gender'] = le.fit_transform(X['gender'])
    X['medical_specialty'] = le.fit_transform(X['medical_specialty'])
    X['payer_code'] = le.fit_transform(X['payer_code'])

    # One-hot encoding on columns with small number of unique values
    low_card_cols = [
        'race', 'admission_type_id', 'discharge_disposition_id',
        'admission_source_id'
    ]
    X = pd.get_dummies(X, columns=low_card_cols, drop_first=True)

    # Scaling numerical variables
    numeric_cols =['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
    'number_diagnoses']
    scaler = MinMaxScaler()
    X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

    return X, y

def split_data(X, y):
    return train_test_split(
        X, y, random_state=0, train_size=0.8
    )

def train_model(X, y):
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X, y)
    return model

def evaluate_model(model, X, y):
    val_prediction = model.predict(X)
    accuracy = accuracy_score(y, val_prediction)
    print(f'Model accuracy: {accuracy}')
    y_pred_proba = model.predict_proba(X)[:,1]
    auc = roc_auc_score(y, y_pred_proba)
    print(f'Model auc score: {auc}')
    return accuracy, auc

def save_model(model, model_path):
    joblib.dump(model, model_path)


In [44]:
# Usage
file_path = '/content/drive/MyDrive/DS & ML/diabetic_data_clean.csv'
data = load_data(file_path)
X, y = preprocess_data(data)
train_X, val_X, train_y, val_y = split_data(X, y)
model = train_model(train_X, train_y)
accuracy, auc = evaluate_model(model, val_X, val_y)
save_model(model, 'diabetes_readmission_model_dt_procedural.pkl')

Model accuracy: 0.9042583595312946
Model auc score: 0.5051150348079422


## **RANDOM FOREST**

In [52]:
# OOP APPROACH
class DiabetesReadmissionPrediction:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None
        self.X = None
        self.y = None
        self.train_X = None
        self.val_X = None
        self.train_y = None
        self.val_y = None
        self.model = None

    def load_data(self):
        self.data = pd.read_csv(self.file_path)

    def preprocess_data(self):
        selected_features = [
            'race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures' , 'num_procedures',
            'num_medications', 'number_diagnoses', 'admission_type_id', 'discharge_disposition_id',
            'admission_source_id','medical_specialty', 'payer_code', 'change', 'diabetesMed',
            'total_visits', 'total_med_types'
        ]
        self.X = self.data[selected_features]
        self.y = self.data[['readmitted']]

        # Label encoding for categorical columns
        le = LabelEncoder()
        self.X['diabetesMed'] = le.fit_transform(self.X['diabetesMed'])
        self.X['change'] = le.fit_transform(self.X['change'])
        self.X['gender'] = le.fit_transform(self.X['gender'])
        self.X['medical_specialty'] = le.fit_transform(self.X['medical_specialty'])
        self.X['payer_code'] = le.fit_transform(self.X['payer_code'])

        # One-hot encoding for low cardinality columns
        low_card_cols = [
            'race', 'admission_type_id', 'discharge_disposition_id',
            'admission_source_id'
        ]
        self.X = pd.get_dummies(self.X, columns=low_card_cols, drop_first=True)

        # Scaling numerical variables
        numeric_cols =['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
        'number_diagnoses']
        scaler = MinMaxScaler()
        self.X[numeric_cols] = scaler.fit_transform(self.X[numeric_cols])

    def split_data(self):
        self.train_X, self.val_X, self.train_y, self.val_y = train_test_split(
            self.X, self.y, random_state=0, train_size=0.8
        )

    def train_model(self):
        self.model = RandomForestClassifier(random_state=0)
        self.model.fit(self.train_X, self.train_y)

    def evaluate_model(self):
        val_prediction = self.model.predict(self.val_X)
        accuracy = accuracy_score(self.val_y, val_prediction)
        print(f'Model accuracy: {accuracy}')
        y_pred_proba = self.model.predict_proba(self.val_X)[:,1]
        auc = roc_auc_score(self.val_y, y_pred_proba)
        print(f'Model auc score: {auc}')
        return accuracy, auc

    def save_model(self, model_path):
        joblib.dump(self.model, model_path)

    def load_model(self, model_path):
        self.model = joblib.load(model_path)


In [53]:
# OOP Usage
readmission_rf = DiabetesReadmissionPrediction('/content/drive/MyDrive/DS & ML/diabetic_data_clean.csv')
readmission_rf.load_data()
readmission_rf.preprocess_data()
readmission_rf.split_data()
readmission_rf.train_model()
accuracy, auc = readmission_rf.evaluate_model()

# Save the model
readmission_rf.save_model('diabetes_readmission_model_rf_oop.pkl')

Model accuracy: 0.909759931408974
Model auc score: 0.6100010679479551


In [50]:
# PROCEDURAL APPROACH
def load_data(file_path):
    return pd.read_csv(file_path)

def preprocess_data(data):
    selected_features = [
        'race', 'gender', 'age', 'time_in_hospital', 'num_lab_procedures' , 'num_procedures',
        'num_medications', 'number_diagnoses', 'admission_type_id', 'discharge_disposition_id',
        'admission_source_id','medical_specialty', 'payer_code', 'change', 'diabetesMed',
        'total_visits', 'total_med_types'
    ]
    X = data[selected_features]
    y = data[['readmitted']]

    # Label encoding
    le = LabelEncoder()
    X['diabetesMed'] = le.fit_transform(X['diabetesMed'])
    X['change'] = le.fit_transform(X['change'])
    X['gender'] = le.fit_transform(X['gender'])
    X['medical_specialty'] = le.fit_transform(X['medical_specialty'])
    X['payer_code'] = le.fit_transform(X['payer_code'])

    # One-hot encoding on columns with small number of unique values
    low_card_cols = [
        'race', 'admission_type_id', 'discharge_disposition_id',
        'admission_source_id'
    ]
    X = pd.get_dummies(X, columns=low_card_cols, drop_first=True)

    # Scaling numerical variables
    numeric_cols =['age','time_in_hospital','num_lab_procedures','num_procedures','num_medications',
    'number_diagnoses']
    scaler = MinMaxScaler()
    X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

    return X, y

def split_data(X, y):
    return train_test_split(
        X, y, random_state=0, train_size=0.8
    )

def train_model(X, y):
    model = RandomForestClassifier(random_state=0)
    model.fit(X, y)
    return model

def evaluate_model(model, X, y):
    val_prediction = model.predict(X)
    accuracy = accuracy_score(y, val_prediction)
    print(f'Model accuracy: {accuracy}')
    y_pred_proba = model.predict_proba(X)[:,1]
    auc = roc_auc_score(y, y_pred_proba)
    print(f'Model auc score: {auc}')
    return accuracy, auc

def save_model(model, model_path):
    joblib.dump(model, model_path)


In [51]:
# PROCEDURAL APPROACH Usage
file_path = '/content/drive/MyDrive/DS & ML/diabetic_data_clean.csv'
data = load_data(file_path)
X, y = preprocess_data(data)
train_X, val_X, train_y, val_y = split_data(X, y)
model = train_model(train_X, train_y)
accuracy, auc = evaluate_model(model, val_X, val_y)
save_model(model, 'diabetes_readmission_model_rf_procedural.pkl')


Model accuracy: 0.909759931408974
Model auc score: 0.6100010679479551


## Model Evaluation Summary

This notebook explored different models to predict if a diabetic patient will be readmitted.

### Metrics Used

*   **Accuracy:** How many predictions were correct overall.
*   **AUC (Area Under the Curve):** How well the model tells apart patients who will be readmitted from those who won't. A score near 0.5 is like guessing, 1.0 is perfect.

### K-Nearest Neighbors (KNN)

*   **Accuracy:** Around 90-91% (looks good).
*   **AUC Score:** Around 0.51-0.56 (very low, close to guessing).
*   **What it means:** The model is good at saying who *won't* be readmitted (the majority). But it's bad at finding the few patients who *will* be readmitted. It's not useful for this problem.

### Decision Tree Classifier

*   **Accuracy:** Around 82-90%.
*   **AUC Score:** Around 0.51 (very low, close to guessing).
*   **What it means:** Similar to KNN, this model also struggles to identify patients who will be readmitted. Not useful in its current form.

### Random Forest Classifier

*   **Accuracy:** Around 91% (looks good).
*   **AUC Score:** Around 0.61 (better, but still needs improvement).
*   **What it means:** This was the best model. It's better at finding patients at risk of readmission than the others. It's a good starting point.

### Key Takeaways

*   **Imbalanced Data:** Many more patients are not readmitted than are. This makes models look good on accuracy, but they fail to find the important 'readmitted' cases.
*   **AUC is Important:** For readmission, AUC is a better measure than accuracy. We need models that can actually identify at-risk patients.
*   **Random Forest is Promising:** It performed best. We can improve it further.

### Next Steps

*   **Address Imbalance:** We should use techniques to help the models learn from the smaller group of 'readmitted' patients.
*   **Improve Random Forest:** We can fine-tune its settings or try new features to boost its AUC score.